In [ ]:
import os
import re
import csv
import pickle as pkl
import numpy as np
import torch
import faiss
import pysbd
import sys
import spacy
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification
import pandas as pd

In [ ]:
ner_model = "../5_train_NER_model/ner_model"
bel_model = "4_fine-tuning/finetuned/final"
idx2cuipath = "../3_pretraining/idx2cui"
umls_csv = "../1_enhance_UMLS/04_ConceptDB/umls-dutch_v1.11_with_drugs_filtered-categories.csv"
idx2cui = pkl.load(open(idx2cuipath, "rb"))

In [ ]:
nlp = spacy.load(
    "nl_core_news_lg",
    disable=["parser", "ner", "lemmatizer"]
)

segmenter = pysbd.Segmenter(language="nl", clean=True)

tokenizer = AutoTokenizer.from_pretrained(
    ner_model, use_fast=True, add_prefix_space=True
)
model = AutoModelForTokenClassification.from_pretrained(ner_model)
model.eval()

In [ ]:
src_parent = "../3_pretraining/sapbert-main"
if src_parent not in sys.path:
    sys.path.append(src_parent)
from src.model_wrapper import Model_Wrapper

In [ ]:
max_length = 50
batch_size = 512
thresh = 0.8

In [ ]:
def normalize_name(name):
    name = name.lower().strip(" ,.;:")
    name = re.sub(r"\s+", " ", name)
    return name
from collections import defaultdict

cui2P = defaultdict(list)
cui2A = defaultdict(list)

with open(umls_csv, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        entry = {
            "name": row["name"],
            "source": row["ontologies"]
        }
        if row["name_status"] == "P":
            cui2P[row["cui"]].append(entry)
        elif row["name_status"] == "A":
            cui2A[row["cui"]].append(entry)

def select_label(cui):
    P_names = cui2P.get(cui)
    if P_names:
        if len(P_names) == 1:
            return P_names[0]["name"]
        for r in P_names:
            if "SNOMEDCT_NL" in r["source"]:
                return r["name"]
        return P_names[0]["name"]

    # with no preferred name take first A
    A_names = cui2A.get(cui)
    if A_names:
        return A_names[0]["name"]


def normalize(s):
    return re.sub(r"\s+", " ", s.lower().strip())

In [ ]:
def fix_bio_sequence(labels):
    fixed = []
    prev = "O"
    for lab in labels:
        if lab == "I" and prev == "O":
            lab = "B"
        fixed.append(lab)
        prev = lab
    return fixed

def extract_entities(tokens, labels):
    entities = []
    buffer = []
    start = None
    for i, (tok, lab) in enumerate(zip(tokens, labels)):
        if lab == "B":
            if buffer:
                entities.append((" ".join(buffer), start, i - 1))
            buffer = [tok]
            start = i
        elif lab == "I" and buffer:
            buffer.append(tok)
        else:
            if buffer:
                entities.append((" ".join(buffer), start, i - 1))
                buffer = []
                start = None
    if buffer:
        entities.append((" ".join(buffer), start, len(tokens) - 1))
    return entities

def is_valid_entity(text):
    t = text.lower().strip(" ,.;:")
    doc = nlp(text)
    if not any(tok.pos_ in {"NOUN", "PROPN"} for tok in doc):
        return False
    return True

def strip_entity(text):
    doc = nlp(text)

    filtered = [
        tok.text
        for tok in doc
        if tok.pos_ in {"NOUN", "PROPN", "ADJ"}
    ]

    if not filtered:
        return None

    return " ".join(filtered)

def normalize_entity(text):
    return re.sub(r"\s+", " ", text.strip(" ,.;:"))

In [ ]:
def getResources(model_directory_path, device="cpu"):
    mw = Model_Wrapper().load_model(
        path=model_directory_path,
        max_length=max_length,
        use_cuda=device=="cuda"
    )
    tokenizer = mw.get_dense_tokenizer()
    model = mw.get_dense_encoder()
    model.eval()
    index = faiss.read_index(f"{model_directory_path}/index_cosine")
    pca = pkl.load(open(f"{model_directory_path}/pca", "rb"))
    return tokenizer, model, index, pca

def get_query_embedding(queries, tokenizer, model, pca, device="cpu"):
    reps = []
    model.to(device)
    model.eval()
    for i in range(0, len(queries), batch_size):
        toks = tokenizer.batch_encode_plus(
            queries[i:i+batch_size],
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        )
        for k,v in toks.items():
            toks[k] = v.to(device)
        with torch.no_grad():
            out = model(**toks)
        reps.append(out[0][:,0,:].cpu().numpy())
    embs = np.vstack(reps).astype(np.float32)
    embs = pca.transform(embs)
    faiss.normalize_L2(embs)  # cosine similarity
    return embs

def query_index(queries, tokenizer, model, index, idx2cui, pca, threshold, device="cpu"):
    embs = get_query_embedding(queries, tokenizer, model, pca, device)
    preds, sims = [], []
    for emb in embs:
        D, I = index.search(np.reshape(emb, (1, emb.shape[0])), k=1)
        score = D[0][0]
        idx = I[0][0]
        sims.append(score)
        if score < threshold:
            preds.append(None)
        else:
            preds.ppend(idx2cui[idx])
    return preds, sims

In [ ]:
label_list = ["O", "B", "I"]
id2label = {i: l for i, l in enumerate(label_list)}
dense_tok, dense_model, index, pca = getResources(bel_model)

Validation and evaluation

In [ ]:
rows = []
uid = 0

txt_path = "random_100.txt"

raw_text = open(txt_path, encoding="utf-8").read()
sentences = segmenter.segment(raw_text)

predictions = []

with torch.no_grad():
    for sent in tqdm(sentences):
        doc = nlp(sent)
        tokens = [t.text for t in doc]
        enc = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=100,
            return_tensors="pt"
        )
        out = model(**enc)
        logits = out.logits[0].cpu().numpy()
        word_ids = enc.word_ids()

        word_to_logits = defaultdict(list)
        for i, w in enumerate(word_ids):
            if w is not None:
                word_to_logits[w].append(logits[i])
        word_logits = [
    np.mean(word_to_logits[i], axis=0)
    for i in sorted(word_to_logits.keys())
]
        labels = [id2label[int(np.argmax(l))] for l in word_logits]
        labels = fix_bio_sequence(labels)
        predictions.append((tokens, labels))

ner_results = []
for (tokens, labels), sent in zip(predictions, sentences):
    ents = extract_entities(tokens, labels)
    ents = [e for e in ents if is_valid_entity(e[0])]
    for text, s, e in ents:
        clean = strip_entity(text)
        if clean is None:
            continue

        text = normalize_entity(clean)
        ner_results.append((sent, text))
for result in ner_results:
    print(result)

mentions = [m for _, m in ner_results]

In [ ]:
preds, sims = query_index(mentions, dense_tok, dense_model, index, idx2cui, pca, threshold=thresh)

for (sent, mention), cui, score in zip(ner_results, preds, sims):
    if cui is None:
        continue
    rows.append({
        "id": uid,
        "sentence": sent,
        "mention": mention,
        "prediction": select_label(cui),
        "cui_prediction": cui,
    })
    uid += 1

for row in rows:
    print(row["prediction"], row["cui_prediction"])

Applying the pipeline to all text files

In [ ]:

input_txt = "TextData/ABEL"
# input_txt = "TextData/IBIS"
# input_txt = "TextData/IBIS_colon"
output_csv = "abel_bel"
output_csv = "ibis_bel"
output_csv = "ibis_colon_bel"
os.makedirs(output_csv, exist_ok=True)

for filename in os.listdir(input_txt):
    print(f"\nProcessing {filename}")
    rows = []
    uid = 0

    txt_path = os.path.join(input_txt, filename)
    raw_text = open(txt_path, encoding="utf-8").read()
    sentences = segmenter.segment(raw_text)

    speaker = "Arts" if "Arts" in filename else "Patient"
    predictions = []

    with torch.no_grad():
        for sent in tqdm(sentences):
            doc = nlp(sent)
            tokens = [t.text for t in doc]
            enc = tokenizer(
                tokens,
                is_split_into_words=True,
                truncation=True,
                max_length=100,
                return_tensors="pt"
            )
            out = model(**enc)
            logits = out.logits[0].cpu().numpy()
            word_ids = enc.word_ids()

            word_to_logits = defaultdict(list)
            for i, w in enumerate(word_ids):
                if w is not None:
                    word_to_logits[w].append(logits[i])
            word_logits = [np.mean(word_to_logits[w], axis=0) for w in range(len(word_to_logits))]
            labels = [id2label[int(np.argmax(l))] for l in word_logits]
            labels = fix_bio_sequence(labels)
            predictions.append((tokens, labels))



    ner_results = []
    for (tokens, labels), sent in zip(predictions, sentences):
        ents = extract_entities(tokens, labels)
        ents = [e for e in ents if is_valid_entity(e[0])]
        for text, s, e in ents:
            clean = strip_entity(text)
            if clean is None:
                continue

            text = normalize_entity(clean)
            ner_results.append((sent, text))
    mentions = [m for _, m in ner_results]

    # Query BEL index
    preds, sims = query_index(mentions, dense_tok, dense_model, index, idx2cui, pca, threshold=thresh)

    for (sent, mention), cui, score in zip(ner_results, preds, sims):
        if cui is None:
            continue
        rows.append({
            "id": uid,
            "sentence": sent,
            "mention": mention,
            "prediction": select_label(cui),
            "cui_prediction": cui,
            "speaker": speaker
        })
        uid += 1


    out_csv = os.path.join(output_csv, os.path.splitext(filename)[0] + ".csv")
    

    columns = [
    "id",
    "sentence",
    "mention",
    "prediction",
    "cui_prediction",
    "speaker"
    ]

    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(out_csv, index=False, encoding="utf-8")